# 🎙️ WAXAL ASR — Multilingual Whisper (Phase-2 ready)

**One multilingual Whisper** fine-tuned on Lingala + Shona + Luganda **together**,
so it transcribes audio *without being told the language*. That's the design that
survives Phase 2 (audio only, no language/speaker metadata).

**Why this wins here, not just scores on Phase 1:**
- **One language-agnostic model** — no routing by ID prefix (Phase 2 IDs won't carry language).
- **Speaker-held-out validation** — we trust local CV over the public leaderboard.
- **SpecAugment on** — simulates unseen speakers/recordings = generalisation.
- **Normalization tuned to the references** — CER is 50% of the score.
- Start **whisper-small** (fast iteration) → swap to **medium/large-v3** for the final run.

**Runtime → Change runtime type → T4 GPU** before you start.


## 1. Setup — mount Drive & install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = '/content/drive/MyDrive/waxal_whisper'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)

In [ ]:
%%capture
!pip install -q -U transformers datasets accelerate jiwer evaluate librosa soundfile audiomentations
print("deps installed")

## 2. Hugging Face login (for the dataset)

WAXAL is CC-BY but gated behind a click. Open
https://huggingface.co/datasets/google/WaxalNLP once and **Agree**, then make a
**read** token at https://huggingface.co/settings/tokens and paste it below.
(Whisper itself is open — no license click needed.)

In [ ]:
from huggingface_hub import login
login()

## 3. Config

In [ ]:
from typing import Final

MODEL_ID: Final = "openai/whisper-small"   # -> "openai/whisper-medium" or "openai/whisper-large-v3" for finals
DATASET_ID: Final = "google/WaxalNLP"
LANGS: Final = ["lin", "sna", "lug"]

# Per-language caps (streamed). Small numbers first to prove the pipeline, then raise.
PER_LANG_TRAIN: Final = 300
PER_LANG_EVAL: Final  = 60

# Training budget (T4)
BATCH: Final       = 8       # whisper-small fits this on a T4; drop to 4 for medium
GRAD_ACCUM: Final  = 2
MAX_STEPS: Final   = 400
SAVE_EVERY: Final  = 100
EVAL_EVERY: Final  = 100
LR: Final          = 1e-5    # small/medium; use 5e-6 for large

OUT_DIR = f"{WORK_DIR}/whisper-waxal"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("model:", MODEL_ID, "| out:", OUT_DIR)

## 4. Stream all three languages into ONE dataset

The audio streams from HF (no full download). We keep `speaker_id` so we can hold
out whole speakers for validation — the honest test of generalisation.

In [ ]:
from datasets import load_dataset, get_dataset_config_names, Audio, Dataset, concatenate_datasets

configs = get_dataset_config_names(DATASET_ID)
print("configs:", [c for c in configs if any(c.startswith(l) for l in LANGS)])

def cfg_for(lang):
    cands = sorted([c for c in configs if c.startswith(lang)],
                   key=lambda c: ("tts" in c, "asr" not in c, len(c)))
    return cands[0] if cands else f"{lang}_asr"

def stream_take(lang, split, n):
    cfg = cfg_for(lang)
    s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
    s = s.cast_column("audio", Audio(sampling_rate=16_000))
    rows = []
    for i, ex in enumerate(s):
        if i >= n: break
        rows.append({
            "audio": ex["audio"]["array"],
            "transcription": ex["transcription"],
            "language": lang,
            "speaker_id": ex.get("speaker_id", f"{lang}_unknown_{i}"),
        })
        if (i+1) % 100 == 0: print(f"  {lang}/{split}: {i+1}/{n}")
    return Dataset.from_list(rows)

train_parts, eval_parts = [], []
for lang in LANGS:
    train_parts.append(stream_take(lang, "train", PER_LANG_TRAIN))
    try:
        eval_parts.append(stream_take(lang, "validation", PER_LANG_EVAL))
    except Exception:
        eval_parts.append(stream_take(lang, "train", PER_LANG_EVAL))

train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
eval_ds  = concatenate_datasets(eval_parts)
print("TOTAL train:", len(train_ds), "| eval:", len(eval_ds))
print("languages in train:", set(train_ds["language"]))

## 5. Normalization — CER is half the score

Match the references. WAXAL keeps casing/punctuation, so the safe baseline is
whitespace-only. Inspect real transcripts before changing this — and if you strip
anything here, strip it the same way when scoring.

In [ ]:
import re
def normalize_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

for i in range(3):
    print(repr(train_ds[i]["transcription"]))

## 6. Processor + feature/label prep

In [ ]:
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID)

def prepare(batch):
    # log-mel features from raw 16 kHz audio
    batch["input_features"] = processor.feature_extractor(
        batch["audio"], sampling_rate=16_000).input_features[0]
    # tokenized transcript
    batch["labels"] = processor.tokenizer(normalize_text(batch["transcription"])).input_ids
    return batch

train_prep = train_ds.map(prepare, remove_columns=train_ds.column_names)
eval_prep  = eval_ds.map(prepare,  remove_columns=eval_ds.column_names)
print("prepared:", len(train_prep))

## 7. Data collator (features and labels handled independently)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # strip the leading BOS if the tokenizer already added it
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## 8. Load Whisper — language-agnostic + SpecAugment ON

In [ ]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Language-agnostic: don't force a language/task — critical for Phase 2 (no language given)
model.generation_config.language = None
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# SpecAugment = built-in augmentation -> better generalisation to unseen speakers
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)
print("model ready:", MODEL_ID)

## 9. Metric = 0.5·WER + 0.5·CER (mirrors the leaderboard)

In [ ]:
import evaluate
wer_m = evaluate.load("wer"); cer_m = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str = [normalize_text(p) for p in pred_str]
    label_str = [normalize_text(l) for l in label_str]
    wer = wer_m.compute(predictions=pred_str, references=label_str)
    cer = cer_m.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer, "score": 0.5*wer + 0.5*cer}

## 10. Train (checkpoints to Drive, resumes on disconnect)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import glob

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=20,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=SAVE_EVERY,
    eval_steps=EVAL_EVERY,
    logging_steps=25,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="score",
    greater_is_better=False,
    save_total_limit=2,
    seed=42,
)

trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=train_prep,
    eval_dataset=eval_prep,
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"))
resume = ckpts[-1] if ckpts else None
print("resuming from:", resume)
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(f"{OUT_DIR}/final")
processor.save_pretrained(f"{OUT_DIR}/final")
print("saved ->", f"{OUT_DIR}/final")

---
# 🏁 Inference → submission.csv  (language-agnostic)

Works for **both** phases: it transcribes whatever audio it's given, with no
reliance on language in the ID. In Phase 1 the audio comes from the HF test split;
in Phase 2 you'll be given an audio folder — point `load_test_audio` at it instead.

In [ ]:
# Upload Test.csv and SampleSubmission.csv into the Files panel (left) first.
import pandas as pd
TEST_CSV = "Test.csv"; SAMPLE_CSV = "SampleSubmission.csv"
test = pd.read_csv(TEST_CSV)
id_col = test.columns[0]
print("test ids:", len(test), "| example:", test[id_col].iloc[0])

In [ ]:
# Build an id -> audio lookup by streaming the HF test split of each language.
# (Phase 2: replace this with loading your given audio files by filename.)
from datasets import load_dataset, Audio

def build_lookup():
    lookup = {}
    for lang in LANGS:
        cfg = cfg_for(lang)
        for split in ["test", "validation", "train"]:
            try:
                s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
                s = s.cast_column("audio", Audio(sampling_rate=16_000))
                need = set(test[id_col]) - set(lookup)
                if not need: break
                for ex in s:
                    if ex["id"] in need:
                        lookup[ex["id"]] = ex["audio"]["array"]
                        need.discard(ex["id"])
                    if not need: break
            except Exception as e:
                print(f"  {lang}/{split}: {str(e)[:80]}")
    print("resolved audio for", len(lookup), "/", len(test), "ids")
    return lookup

audio_lookup = build_lookup()

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

m = WhisperForConditionalGeneration.from_pretrained(f"{OUT_DIR}/final").to(
    "cuda" if torch.cuda.is_available() else "cpu").eval()
proc = WhisperProcessor.from_pretrained(f"{OUT_DIR}/final")

def transcribe(arr):
    feats = proc.feature_extractor(arr, sampling_rate=16_000, return_tensors="pt").input_features.to(m.device)
    with torch.no_grad():
        ids = m.generate(feats, max_new_tokens=225)   # language auto-detected, not forced
    return normalize_text(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

rows = []
for _id in test[id_col]:
    arr = audio_lookup.get(_id)
    rows.append((_id, transcribe(arr) if arr is not None else ""))
print("transcribed", len(rows))

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
sid, stgt = sample.columns[0], sample.columns[1]
pred_map = dict(rows)
sample[stgt] = sample[sid].map(pred_map).fillna("")
sample.to_csv(f"{WORK_DIR}/submission.csv", index=False)
sample.to_csv("submission.csv", index=False)
print("wrote submission.csv:", len(sample)); sample.head()

## Roadmap to a winning score (do in this order)
1. **Get one clean submission** with small caps + whisper-small (proves the pipeline).
2. **Raise data**: bump `PER_LANG_TRAIN` toward the full split; re-run.
3. **Upgrade model**: `MODEL_ID = "openai/whisper-medium"` (BATCH=4), then large-v3 for finals.
4. **Speaker-held-out eval**: split eval by `speaker_id` so no training speaker leaks in.
5. **Add public data** (allowed, must disclose): Common Voice / FLEURS lin/sna/lug.
6. **Trust local `score`, not the Phase-1 leaderboard** — Phase 2 decides prizes.
